In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Embedding, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [2]:
VOCAB_SIZE  = 10_000
MAX_LEN     = 200
EMBED_DIM   = 128
EPOCHS      = 5
BATCH_SIZE  = 128
OOV_TOKEN   = "<OOV>"

In [3]:
dataset, _ = tfds.load("imdb_reviews", with_info=True, as_supervised=True)

train_sentences, train_labels = [], []
test_sentences,  test_labels  = [], []

for text, label in dataset["train"]:
    train_sentences.append(text.numpy().decode("utf-8"))
    train_labels.append(label.numpy())

for text, label in dataset["test"]:
    test_sentences.append(text.numpy().decode("utf-8"))
    test_labels.append(label.numpy())

train_labels = np.array(train_labels)
test_labels  = np.array(test_labels)

print(f"Train: {len(train_sentences)} samples")
print(f"Test : {len(test_sentences)} samples")


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DPC4RY_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DPC4RY_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DPC4RY_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
Train: 25000 samples
Test : 25000 samples


In [4]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(train_sentences)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_sentences),
    maxlen=MAX_LEN, dtype="int32", padding="post", truncating="post"
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_sentences),
    maxlen=MAX_LEN, dtype="int32", padding="post", truncating="post"
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

X_train shape: (25000, 200)
X_test  shape: (25000, 200)


In [5]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, train_labels,
    validation_data=(X_test, test_labels),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.7242 - loss: 0.5265 - val_accuracy: 0.8466 - val_loss: 0.3487
Epoch 2/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8910 - loss: 0.2774 - val_accuracy: 0.8617 - val_loss: 0.3160
Epoch 3/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9542 - loss: 0.1367 - val_accuracy: 0.8688 - val_loss: 0.3286
Epoch 4/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9880 - loss: 0.0499 - val_accuracy: 0.8666 - val_loss: 0.3909
Epoch 5/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9981 - loss: 0.0154 - val_accuracy: 0.8655 - val_loss: 0.4611


In [6]:
loss, acc = model.evaluate(X_test, test_labels, verbose=0)
print(f"\nTest accuracy : {acc:.4f}  |  Loss : {loss:.4f}")



Test accuracy : 0.8655  |  Loss : 0.4611


In [7]:
SAMPLE_REVIEWS = [
    "This film was an absolute masterpiece. The acting was superb and the story "
    "kept me hooked till the very end. Highly recommend!",

    "Terrible movie. The plot made no sense, the acting was wooden, and I nearly "
    "fell asleep halfway through. Complete waste of time.",

    "Some scenes were genuinely moving and the cinematography was beautiful, but "
    "the pacing dragged badly in the second half and the ending felt rushed.",

    "Not the worst film I have ever seen, but definitely not great either. "
    "The lead actor tried his best with a very weak script.",

    "One of the best films I have ever watched. Every frame is crafted with care "
    "and the performances are nothing short of extraordinary.",
]

seqs   = [[t for t in seq if t is not None]
          for seq in tokenizer.texts_to_sequences(SAMPLE_REVIEWS)]
padded = pad_sequences(seqs, maxlen=MAX_LEN, dtype="int32", padding="post", truncating="post")
preds  = model.predict(padded, verbose=0).flatten()

print("\n" + "=" * 65)
print("  SAMPLE INFERENCE OUTPUTS")
print("=" * 65)

for i, (review, score) in enumerate(zip(SAMPLE_REVIEWS, preds), 1):
    label      = "POSITIVE" if score >= 0.5 else "NEGATIVE"
    confidence = score if score >= 0.5 else 1 - score
    bar        = "█" * int(confidence * 20) + "░" * (20 - int(confidence * 20))

    print(f"\n[{i}] {review[:75]}{'...' if len(review) > 75 else ''}")
    print(f"     Label      : {label}")
    print(f"     Raw score  : {score:.4f}")
    print(f"     Confidence : {bar}  {confidence * 100:.1f}%")

print("\n" + "=" * 65)


  SAMPLE INFERENCE OUTPUTS

[1] This film was an absolute masterpiece. The acting was superb and the story ...
     Label      : POSITIVE
     Raw score  : 1.0000
     Confidence : ███████████████████░  100.0%

[2] Terrible movie. The plot made no sense, the acting was wooden, and I nearly...
     Label      : NEGATIVE
     Raw score  : 0.0000
     Confidence : ███████████████████░  100.0%

[3] Some scenes were genuinely moving and the cinematography was beautiful, but...
     Label      : NEGATIVE
     Raw score  : 0.0037
     Confidence : ███████████████████░  99.6%

[4] Not the worst film I have ever seen, but definitely not great either. The l...
     Label      : NEGATIVE
     Raw score  : 0.0000
     Confidence : ███████████████████░  100.0%

[5] One of the best films I have ever watched. Every frame is crafted with care...
     Label      : POSITIVE
     Raw score  : 0.9996
     Confidence : ███████████████████░  100.0%

